# Conversational Mode

Conversational mode is a behavioral pattern where the agent's primary activity is dialogue - asking questions, exploring ideas, clarifying ambiguity, and providing nuanced advice through back-and-forth exchange. The conversation itself is the value delivered; no specific task deliverable is required.

Unlike a basic chatbot (which is reactive and one-shot), conversational mode is active and directive: the agent steers the conversation toward clarity, insight, and useful outcomes. It manages dialogue as its core responsibility rather than treating conversation as a side-channel to task execution.

This mode is classified as dialogue-focused engagement. It can combine with any autonomy level:
- Conversational + chat: Pure dialogue, no tool use.
- Conversational + copilot: Agent gathers context via read-only tools to inform the conversation.
- Conversational + supervised: Agent can take actions mid-conversation but pauses for confirmation.

In [1]:
import os
import json
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

### Initialize the language model
Conversational mode benefits from a slightly higher temperature than task-execution modes. A temperature of `0.7` allows the agent to vary its phrasing naturally across turns - essential for dialogue that doesn't feel robotic - while still staying focused on the conversation goal.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

print("LLM initialized for conversational mode.")

LLM initialized for conversational mode.


## Tracking conversation state
What separates a conversational agent from a simple prompt-response loop is state. The agent needs to remember what has been established, what remains unclear, and what preferences the human has revealed. Without this, every response starts from scratch - the agent cannot build toward a conclusion.

We track four key elements:
- `established_facts`: What has been agreed or confirmed.
- `open_questions`: What remains unclear or unresolved.
- `human_preferences`: Constraints, values, or priorities the human has revealed.
- `conversation_phase`: Which stage the dialogue is in (exploring → clarifying → advising → concluding).

In [3]:
@dataclass
class ConversationState:
    """The agent's running model of a conversational session."""
    topic: str                    # domain or subject of the conversation
    goal: str                     # what the conversation is meant to achieve for the human
    established_facts: list = field(default_factory=list)   # confirmed or agreed information
    open_questions: list = field(default_factory=list)      # unresolved questions to address
    human_preferences: dict = field(default_factory=dict)   # revealed constraints and values
    conversation_phase: str = "exploring"  # exploring → clarifying → advising → concluding
    turn_count: int = 0           # incremented before each response is generated

`ConversationState` is intentionally separate from the message history. The history is a raw transcript - every word that was exchanged. The state is a *synthesis* of that transcript: what matters, what was resolved, and what still needs attention. The strategy selection logic reads the state, not the raw history, which keeps it fast and predictable.

In [4]:
# Create a state object and inspect it before any conversation begins
state = ConversationState(
    topic="Career transition from software engineering to management",
    goal="Help the professional decide whether to accept a management promotion"
)

# Pre-populate some open questions — in a live session these emerge from the first user message
state.open_questions = [
    "Does the person value technical work or leadership?",
    "What are the financial implications?",
    "What happens to technical skills over time in management?",
    "What is the company culture around management?",
    "Is the person ready for leadership responsibilities?",
]

print(f"Topic    : {state.topic}")
print(f"Goal     : {state.goal}")
print(f"Phase    : {state.conversation_phase}")
print(f"Open q's : {len(state.open_questions)}")
for q in state.open_questions:
    print(f"  - {q}")

Topic    : Career transition from software engineering to management
Goal     : Help the professional decide whether to accept a management promotion
Phase    : exploring
Open q's : 5
  - Does the person value technical work or leadership?
  - What are the financial implications?
  - What happens to technical skills over time in management?
  - What is the company culture around management?
  - Is the person ready for leadership responsibilities?


## Step 1: Selecting a strategy
At the start of every turn the agent makes one decision before generating any text: what *kind* of move should this response be? This decision - strategy selection - is what distinguishes a purposeful conversational agent from a plain chatbot that simply responds to whatever was last said.

Four strategies cover the full lifecycle of a productive dialogue. When many questions are open, the agent should *clarify* - ask exactly one focused question rather than attempting to address everything at once. When a few questions remain, it should *explore* - share its current understanding and invite the human to refine it. Periodically, it should *summarize* - check that both parties share the same picture of what has been established. And once enough context is in place, it should *advise* - give a direct, substantive recommendation rather than continuing to ask questions.

The critical rule for clarification is worth stating explicitly: one question per turn. Asking multiple questions at once overwhelms the human and signals that the agent has not prioritized. If there are five open questions, the agent's job is to decide which one matters most right now - not to ask all five.

In [5]:
def select_strategy(state: ConversationState) -> str:
    """Select the appropriate conversational move based on the current state.

    Args:
        state: The current conversation state.

    Returns:
        Strategy name: 'clarify', 'explore', 'summarize', or 'advise'.
    """
    open_q = len(state.open_questions)   # how many unresolved questions remain
    turn = state.turn_count              # how many turns have been completed

    if open_q > 4:
        return "clarify"    # too many unknowns — pick one and ask it
    elif open_q > 2:
        return "explore"    # probe the most important remaining open question
    elif turn > 1 and turn % 4 == 0:
        return "summarize"  # every 4 turns, pause to confirm shared understanding
    else:
        return "advise"     # enough context — give direct, substantive advice

The strategy thresholds are deliberately simple integer comparisons rather than LLM inference. Keeping control flow deterministic makes the agent's behavior predictable and debuggable - a developer can look at the state and know exactly which strategy will be chosen without running the model. The LLM's role is generating the *content* of the response, not deciding *what kind* of response to make. Separating these two concerns is what makes the agent's behavior tunable: adjusting a threshold is a one-line change, not a prompt-engineering exercise.

The `turn % 4 == 0` condition fires a summary every four turns regardless of the open-question count. Periodic alignment checks prevent long conversations from drifting - both parties accumulate assumptions about what was agreed, and those assumptions sometimes diverge silently.

In [6]:
# Show how the selected strategy changes as open questions reduce across turns
print("Strategy selection at different state snapshots:")
print("-" * 52)
print(f"  {'Open questions':>15}  {'Turn':>5}  {'Strategy':<10}")
print("-" * 52)

snapshots = [
    (5, 0),   # many questions, conversation just started
    (3, 2),   # moderate questions, a few turns in
    (2, 4),   # few questions, turn 4 — triggers summarize
    (2, 5),   # same question count, different turn — advise kicks in
    (1, 6),   # one question left, turn 6
    (0, 7),   # all questions resolved
]

for n_questions, turn in snapshots:
    # Temporarily set state values to simulate each snapshot
    state.open_questions = ["placeholder"] * n_questions
    state.turn_count = turn
    strategy = select_strategy(state)
    print(f"  {n_questions:>15}  {turn:>5}  {strategy.upper():<10}")

Strategy selection at different state snapshots:
----------------------------------------------------
   Open questions   Turn  Strategy  
----------------------------------------------------
                5      0  CLARIFY   
                3      2  EXPLORE   
                2      4  SUMMARIZE 
                2      5  ADVISE    
                1      6  ADVISE    
                0      7  ADVISE    


## Step 2: Building the system prompt
Rather than using a single fixed system prompt across all turns, conversational mode injects the current state and selected strategy into the prompt before each response. This is what makes the agent context-aware: it does not merely have access to the message history - it is explicitly told what has been confirmed, what is still open, and what kind of move to make next.

The per-strategy instruction is the most important part of the prompt. It tells the agent not just what to talk about but *how to structure its response*. The clarify instruction, for instance, forbids asking multiple questions; the advise instruction demands a direct recommendation rather than endless caveats. Without these guardrails, the model tends to default to hedging, re-asking questions that have already been answered, or producing generic advice that ignores what the conversation has established.

Keeping the prompt construction as a standalone function - separate from the LLM call - makes the prompt inspectable. During development, printing the prompt before running the model is the fastest way to diagnose why an agent behaved unexpectedly.

In [7]:
# Per-strategy instructions that shape the specific move for this turn
STRATEGY_INSTRUCTIONS = {
    "clarify": (
        "Ask exactly ONE focused clarifying question. "
        "Choose the question whose answer will most reduce uncertainty. "
        "Do not ask multiple questions — choose the single most important one."
    ),
    "explore": (
        "Share your current understanding of the most important open question "
        "and invite the human to refine or correct it. "
        "Be specific about what you understand so far."
    ),
    "summarize": (
        "Briefly summarize what has been established in this conversation so far. "
        "State the key facts and the human's main preferences or constraints. "
        "Then ask: 'Does this capture where we are?' before continuing."
    ),
    "advise": (
        "Provide substantive, specific advice based on what has been established. "
        "Be direct — give a clear recommendation, not endless caveats. "
        "Acknowledge trade-offs explicitly. Where you are uncertain, say so briefly "
        "and then still provide your best recommendation."
    ),
}


def build_system_prompt(state: ConversationState, strategy: str) -> str:
    """Construct a dynamic system prompt from the current state and selected strategy.

    Args:
        state: The current conversation state.
        strategy: The strategy selected for this turn.

    Returns:
        A complete system prompt string ready for use as a SystemMessage.
    """
    # Format established facts as a readable list, or a placeholder if none yet
    facts_text = (
        "\n".join(f"  - {f}" for f in state.established_facts)
        if state.established_facts else "  None yet"
    )
    # Format open questions the same way
    questions_text = (
        "\n".join(f"  - {q}" for q in state.open_questions)
        if state.open_questions else "  None remaining"
    )

    return f"""You are a thoughtful conversation partner operating in Conversational Mode.

CONVERSATION GOAL: {state.goal}
TOPIC: {state.topic}
CURRENT PHASE: {state.conversation_phase}

ESTABLISHED FACTS:
{facts_text}

OPEN QUESTIONS:
{questions_text}

HUMAN PREFERENCES: {state.human_preferences if state.human_preferences else 'None revealed yet'}

RESPONSE STRATEGY: {strategy.upper()}
{STRATEGY_INSTRUCTIONS[strategy]}

Respond naturally and conversationally. Keep responses concise (3-5 sentences) unless the strategy calls for a summary."""

The `STRATEGY_INSTRUCTIONS` dict is defined at module level rather than inside the function because it is a static mapping that does not depend on any runtime state. Keeping it outside the function avoids recreating the dict on every call and makes the instructions easy to inspect and tune independently.

The prompt embeds the full state - facts, open questions, and preferences - rather than just the strategy label. This matters because the LLM needs to know *which* open question to ask about, not merely *that* it should ask a question. Without the question list, a "clarify" instruction produces generic questions rather than targeted ones.

In [8]:
# Reset the state to a concrete configuration and inspect the generated prompt
state.open_questions = [
    "Does the person value technical work or leadership?",
    "What are the financial implications?",
    "What happens to technical skills over time in management?",
]
state.established_facts = ["6 years of engineering experience", "Manager believes they are ready"]
state.turn_count = 2

strategy = select_strategy(state)   # 3 open questions → explore
prompt = build_system_prompt(state, strategy)

print(f"Strategy selected: {strategy.upper()}")
print("\nGenerated system prompt:")
print("-" * 55)
print(prompt)

Strategy selected: EXPLORE

Generated system prompt:
-------------------------------------------------------
You are a thoughtful conversation partner operating in Conversational Mode.

CONVERSATION GOAL: Help the professional decide whether to accept a management promotion
TOPIC: Career transition from software engineering to management
CURRENT PHASE: exploring

ESTABLISHED FACTS:
  - 6 years of engineering experience
  - Manager believes they are ready

OPEN QUESTIONS:
  - Does the person value technical work or leadership?
  - What are the financial implications?
  - What happens to technical skills over time in management?

HUMAN PREFERENCES: None revealed yet

RESPONSE STRATEGY: EXPLORE
Share your current understanding of the most important open question and invite the human to refine or correct it. Be specific about what you understand so far.

Respond naturally and conversationally. Keep responses concise (3-5 sentences) unless the strategy calls for a summary.


## Step 3: Processing a conversation turn
With state, strategy selection, and prompt construction in place, a single conversation turn reduces to three steps: increment the turn counter, build the prompt, and invoke the LLM. The message history is passed in and an extended copy is returned - the function does not mutate any shared list, which keeps each turn independently repeatable if needed for debugging.

A separate `update_state` function handles the changes to established facts, open questions, and preferences that follow each turn. Keeping state updates explicit and separate from the response generation makes the notebook easier to follow: one function asks and responds, the other records what was learned. In a production system, the state update step would itself be driven by an LLM call that reads the latest exchange and extracts new facts - but for this notebook, the updates are applied manually to keep the focus on the conversation logic itself.

In [9]:
def respond(state: ConversationState, history: list, user_message: str) -> tuple:
    """Process one conversation turn and return the agent's reply and updated history.

    Args:
        state: The current conversation state (turn_count is mutated in place).
        history: The full message history from previous turns.
        user_message: The latest message from the human.

    Returns:
        Tuple of (agent_reply: str, updated_history: list, strategy_used: str).
    """
    state.turn_count += 1              # increment before strategy selection — turn count affects the choice

    strategy = select_strategy(state)  # decide what kind of move to make this turn
    system_prompt = build_system_prompt(state, strategy)

    # Assemble full message list: system prompt + prior history + new user message
    messages = (
        [SystemMessage(content=system_prompt)]
        + history
        + [HumanMessage(content=user_message)]
    )

    resp = llm.invoke(messages)
    agent_reply = resp.content.strip()

    # Return a new extended history — do not mutate the incoming list
    updated_history = history + [
        HumanMessage(content=user_message),
        AIMessage(content=agent_reply),
    ]

    return agent_reply, updated_history, strategy


def update_state(
    state: ConversationState,
    new_facts: list = None,
    answered: list = None,
    new_questions: list = None,
    preferences: dict = None,
) -> None:
    """Update conversation state with information extracted from the latest turn.

    Args:
        state: The state object to update in place.
        new_facts: Facts to add to established_facts.
        answered: Questions to remove from open_questions (they have been resolved).
        new_questions: New questions to add to open_questions.
        preferences: Key-value pairs to merge into human_preferences.
    """
    if new_facts:
        state.established_facts.extend(new_facts)
    if answered:
        # Keep only questions that are not in the answered list
        state.open_questions = [q for q in state.open_questions if q not in answered]
    if new_questions:
        state.open_questions.extend(new_questions)
    if preferences:
        state.human_preferences.update(preferences)

`respond` returns a new history list rather than mutating the one passed in. This makes the function safe to call multiple times from the same starting point without side effects - useful during development when re-running cells to inspect agent behavior at specific turns.

The `update_state` function uses a list comprehension to filter answered questions: it keeps every question that is *not* in the `answered` list. This comparison is string-based, so the exact wording used in `answered` must match what was originally added to `open_questions`. In a production system, fuzzy matching or an LLM-based deduplication step would replace this exact-match filter.

## Demo: Career decision coaching conversation
The scenario below simulates a software engineer seeking guidance on whether to accept a management promotion. The human's side of the conversation is pre-scripted, while the agent responds dynamically using the strategy-selection and prompt-building logic developed above.

State updates are applied after each turn to reflect what was learned from the exchange. In a production system these updates would be generated automatically - an LLM call reads the latest exchange and extracts new facts, answered questions, and revealed preferences. Here they are applied manually so the notebook can show, turn by turn, how the strategy shifts as the agent learns more about the situation.

In [10]:
# Fresh state for the demo — no facts established, several open questions
state = ConversationState(
    topic="Career transition from software engineering to management",
    goal="Help the professional decide whether to accept a management promotion"
)
update_state(state, new_questions=[
    "Does the person value technical work or leadership?",
    "What are the financial implications of the role?",
    "What happens to technical skills over time in management?",
    "What is the company culture around management?",
    "Is the person ready for leadership responsibilities?",
])

history = []  # message history grows with each turn

conversation_turns = [
    "I've been offered a promotion to engineering manager. I'm not sure whether to take it.",
    "I've been an engineer for 6 years. I enjoy the technical work but my manager says I'm ready to lead.",
    "The pay increase is about 20%. But I'm worried I'll lose my technical edge over time.",
    "I think I'd miss solving hard problems myself. Though I do enjoy mentoring junior engineers.",
    "What would you recommend?",
]

print("=" * 65)
print("CAREER DECISION COACHING CONVERSATION")
print("=" * 65)

for i, user_msg in enumerate(conversation_turns, 1):
    agent_reply, history, strategy = respond(state, history, user_msg)

    print(f"\nTurn {i} | Strategy: {strategy.upper()} | Open questions: {len(state.open_questions)}")
    print(f"Human : {user_msg}")
    print(f"Agent : {agent_reply}")

    # Update state with what was learned in this turn
    if i == 2:
        update_state(
            state,
            new_facts=["6 years of engineering experience", "Manager believes they are ready"],
            answered=["Is the person ready for leadership responsibilities?"]
        )
    elif i == 3:
        update_state(
            state,
            new_facts=["20% pay increase offered"],
            answered=["What are the financial implications of the role?"],
            preferences={"concerned_about_technical_skills": True}
        )
    elif i == 4:
        update_state(
            state,
            new_facts=["Values solving hard technical problems", "Enjoys mentoring junior engineers"],
            answered=["Does the person value technical work or leadership?"]
        )
    elif i == 5:
        # Move to the advising phase once the human explicitly asks for a recommendation
        state.conversation_phase = "advising"

print("\n" + "=" * 65)
print("FINAL STATE SUMMARY")
print("=" * 65)
print(f"Phase              : {state.conversation_phase}")
print(f"Established facts  : {state.established_facts}")
print(f"Remaining questions: {state.open_questions}")
print(f"Preferences        : {state.human_preferences}")
print(f"Total turns        : {state.turn_count}")

CAREER DECISION COACHING CONVERSATION

Turn 1 | Strategy: CLARIFY | Open questions: 5
Human : I've been offered a promotion to engineering manager. I'm not sure whether to take it.
Agent : That's a significant opportunity! To help you think it through, can you share what you value more in your career right now: continuing to work on technical projects or stepping into a leadership role?

Turn 2 | Strategy: CLARIFY | Open questions: 5
Human : I've been an engineer for 6 years. I enjoy the technical work but my manager says I'm ready to lead.
Agent : It sounds like you have a solid foundation in technical work, but transitioning to leadership can be quite different. Have you considered what aspects of leadership excite or concern you? Understanding your feelings about leading a team might help clarify whether this promotion aligns with your career goals.

Turn 3 | Strategy: EXPLORE | Open questions: 4
Human : The pay increase is about 20%. But I'm worried I'll lose my technical edge over

The strategy label printed beside each turn shows the agent's behavior shifting as the conversation progresses. Early turns, when many questions are open, produce `CLARIFY` or `EXPLORE` responses - focused questions, one at a time. As facts accumulate and open questions reduce, the agent transitions to `ADVISE` and gives a direct recommendation rather than asking another question. The strategy column makes this shift auditable: if the agent asked one more clarifying question when it should have advised, the state snapshot at that turn shows exactly why.

## The funnel pattern
One of the most reliable structures for conversational mode is the funnel pattern: start broad to understand the full situation, then progressively narrow toward the single constraint that most needs addressing. The funnel moves through four turns in sequence - broad context, core challenge, specific blocker, targeted advice — rather than letting the conversation wander.

The funnel earns the right to advise. Jumping straight to a recommendation before understanding the situation produces generic advice that misses the real constraint. A developer whose API is slow could be dealing with a missing index, an N+1 query, a network bottleneck, or a caching gap - all of which call for completely different fixes. The funnel forces the agent to discover which one applies before prescribing anything.

```
Turn 1:  "Tell me about the current situation"       → Broad context
Turn 2:  "Which aspect is most pressing right now?" → Identify core challenge
Turn 3:  "What has prevented solving this so far?"  → Expose specific blocker
Turn 4+: Advise directly on the specific blocker     → Targeted recommendation
```

In [11]:
def run_funnel_conversation(topic: str, goal: str, user_turns: list) -> None:
    """Run the funnel pattern on a pre-scripted conversation and print the dialogue.

    Args:
        topic: The subject domain of the conversation.
        goal: What the conversation is trying to achieve.
        user_turns: Pre-scripted user messages in order.
    """
    system_prompt = f"""You are a strategic advisor using the Funnel Pattern.

GOAL: {goal}
TOPIC: {topic}

Funnel structure — progressively narrow with each turn:
- Turn 1: Ask one broad question to understand the full situation
- Turn 2: Identify the single most pressing challenge
- Turn 3: Explore the specific constraint blocking progress
- Turn 4+: Provide targeted advice on the specific constraint

Keep each response to 2-3 sentences. Ask only ONE question per turn."""

    # Phase labels by turn number — turns beyond 3 default to "Advise"
    phase_labels = {1: "Broad", 2: "Narrow", 3: "Specific"}

    history = []
    print(f"FUNNEL CONVERSATION: {topic}")
    print("-" * 55)

    for i, user_msg in enumerate(user_turns, 1):
        history.append(HumanMessage(content=user_msg))

        # The system prompt plus full history gives the model full context for each turn
        messages = [SystemMessage(content=system_prompt)] + history
        response = llm.invoke(messages)
        history.append(AIMessage(content=response.content))

        # Determine phase label from the turn index; default to Advise once past the funnel
        phase = phase_labels.get(i, "Advise")
        print(f"\nTurn {i} [{phase}]")
        print(f"Human: {user_msg}")
        print(f"Agent: {response.content}")

In [12]:
# Demonstrate the funnel pattern on a technical troubleshooting conversation
run_funnel_conversation(
    topic="API performance improvement",
    goal="Help the engineer identify and address the root cause of API slowdowns",
    user_turns=[
        "Our API has been slow lately and users are complaining.",
        "The slowness is worst on the search endpoint, especially for large result sets.",
        "We haven't added any indexes since we launched, and the table has grown to 5 million rows.",
    ]
)

FUNNEL CONVERSATION: API performance improvement
-------------------------------------------------------

Turn 1 [Broad]
Human: Our API has been slow lately and users are complaining.
Agent: What specific metrics or user feedback are you seeing that indicate the API's slow performance?

Turn 2 [Narrow]
Human: The slowness is worst on the search endpoint, especially for large result sets.
Agent: What do you believe is the primary factor contributing to the search endpoint's slow performance with large result sets?

Turn 3 [Specific]
Human: We haven't added any indexes since we launched, and the table has grown to 5 million rows.
Agent: What challenges are you facing in implementing indexing on the database table to improve the performance of the search endpoint?


The funnel prevents the agent from jumping to "add an index" in Turn 1 when the real situation might be network latency, query design, caching gaps, or connection pool exhaustion - all of which call for different fixes. By Turn 3, the specific constraint has been surfaced through dialogue. The advice that follows is precise rather than a list of generic database-optimization suggestions that may not even apply.

## When to use conversational mode
Conversational mode is the right choice when the *answer* cannot be determined without first understanding the *situation* more deeply. Use it when:

| Situation | Why Conversational Mode Fits |
|-----------|-----------------------------|
| Problem not yet fully defined | Human needs help clarifying what they actually want |
| Decision involves trade-offs | No single right answer; human must weigh competing values |
| Domain expertise needed | Human needs to understand a complex space before deciding |
| Creative exploration | Brainstorming or design thinking where the space is open |
| Emotional context matters | Human needs to feel heard before they can act on advice |
| High stakes / irreversible | Thorough dialogue before any action is taken |

**When NOT to use conversational mode**: If the task has a clear, bounded deliverable, use task execution Mode instead. Conversational mode applied to a clear task just adds unnecessary friction.

- **Conversational mode is active, not reactive**: The agent steers the dialogue toward a goal rather than simply responding to whatever the human says.
- **Strategy selection separates good conversational agents from chatbots**: Deciding *whether* to clarify, explore, summarize, or advise - based on state, not just the latest message - is what makes the agent purposeful.
- **One question per turn is a hard rule**: Asking multiple questions at once overwhelms the human and signals poor prioritization. Choose the single question whose answer reduces uncertainty the most.
- **The funnel pattern earns the right to advise**: By progressively narrowing from broad context to specific constraint, the agent ensures advice is targeted rather than generic.
- **Conversational mode pairs with any autonomy level**: The behavioral mode (dialogue management) is orthogonal to the autonomy mode (how much the agent can act independently).